# Human In The Loop

Clean LangGraph interrupt/resume example. Run cells from top to bottom after restarting the kernel.

In [ ]:
from pathlib import Path
import os
import sys

project_root = Path.cwd().resolve()
while not (project_root / "pyproject.toml").exists():
    if project_root == project_root.parent:
        raise RuntimeError("Could not find project root with pyproject.toml")
    project_root = project_root.parent

os.chdir(project_root)
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv

load_dotenv(project_root / ".env", override=True)

print("Project root:", project_root)
print("GROQ_API_KEY loaded:", bool(os.getenv("GROQ_API_KEY")))


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile")
llm


In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, interrupt


class State(TypedDict):
    messages: Annotated[list, add_messages]


@tool
def human_assistance(query: str) -> str:
    """Request assistance from a human expert."""
    human_response = interrupt({"query": query})
    return human_response["data"]


tools = [human_assistance]
llm_with_tools = llm.bind_tools(tools)


def chatbot(state: State):
    message = llm_with_tools.invoke(state["messages"])
    return {"messages": [message]}


graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools))
graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)


In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:
    print("Graph image skipped:", exc)


## 1. Start The Request

This should pause at the `human_assistance` interrupt. Keep the same `thread_id` for resume.

In [ ]:
config = {"configurable": {"thread_id": "human-demo-1"}}

user_input = [
    (
        "user",
        "I need some expert guidance and assistance for building an AI agent. Could you request assistance for me?",
    )
]

events = graph.stream(
    {"messages": user_input},
    config,
    stream_mode="values",
)

for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()


In [ ]:
snapshot = graph.get_state(config)
print("Next node waiting:", snapshot.next)
print("Interrupts:", snapshot.interrupts)

if not snapshot.next:
    print("No interrupt is waiting. Rerun the previous request cell before resuming.")


## 2. Resume With Human Answer

Run this only after the previous cell pauses and `snapshot.next` is not empty.

In [ ]:
human_response = (
    "We, the experts are here to help! We'd recommend you check out LangGraph to build your agent."
    " It's much more reliable and extensible than simple autonomous agents."
)

events = graph.stream(
    Command(resume={"data": human_response}),
    config,
    stream_mode="values",
)

for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()
